# DS Processing

**Data: Campi Flegrei.** Sentinel-1A IW, descending track 22, VV, 92 dates from 2019-01-02 to 2021-12-29 (every 12 days, reference 2020-07-07), cropped to the caldera (Pozzuoli and the Gulf of Pozzuoli) at full resolution. A bell-shaped uplift of tens of centimetres in the centre; the town gives many PS. The data and how they were made are described in `data/CampiFlegrei/README.md`.

The thresholds below are starting values; look at the figures and adjust them to the data.


In this tutorial, we demostrate how to do DS processing with the decorrelation CLI interface.
The input data for DS processing is prepared in [Load Data](./01_load.ipynb).

In [ ]:
!ls ./load_data

## SHP selection

Here we use two sample Kolmogorov-Smirnov (KS) test for Spatially Homogenious Pixel (SHP) identification.

The output `pvalue` is the P-value for this test.
The smaller the p-value, the more likely the two sample are from same distribution, i.e.,
the more likely the two pixels are SHP to each other.

In [ ]:
import numpy as np
import moraine.cli as mc
import moraine as mr

In [ ]:
rslc = './load_data/rslc.zarr'
pvalue = './ds_processing/gix/ras_pvalue.zarr'
az_half_win = 5
r_half_win = 5
method = 'ks'

In [ ]:
logger = mc.get_logger()

In [ ]:
mc.shp_test(rslc,pvalue,
            az_half_win=az_half_win,r_half_win=r_half_win)

Then we select SHP by setting a threshold on this Pvalue:

In [ ]:
import zarr
from matplotlib import pyplot as plt
import colorcet

In [ ]:
is_shp = './ds_processing/gix/ras_is_shp.zarr'
shp_num = './ds_processing/gix/ras_shp_num.zarr'
p_max = 0.05

In [ ]:
mc.select_shp(pvalue,is_shp,shp_num,p_max=p_max)

The output `is_shp` indicate wheather a pixel in the window is a SHP to its center pixel.

In [ ]:
shp_num_zarr = zarr.open(shp_num,mode='r')
mc.view(shp_num, label='number of SHP')

## DS candidates selection

In [ ]:
ds_can_gix = './ds_processing/gix/ds_can_gix.zarr'
is_ds_can = './ds_processing/gix/ras_is_ds_can.zarr'

In [ ]:
nshp_min = 50   # DS candidates: at least this number of SHP
mc.pc_logic_ras(shp_num,ds_can_gix,f'ras>={nshp_min}',chunks=200000)
mc.gix2bool(ds_can_gix, is_ds_can, shape= shp_num_zarr.shape)

`ds_can_idx` is a int array with shape of (num_of_DS_can, 2). It indicate the position of DS candidate in radar coordinate.

In [ ]:
mc.view(is_ds_can, label='DS candidates')

## Coherence matrix estimation

The coherence matrix estimation is kind of complicated as it involves both raster and point cloud data. We also want to save the output point cloud data in the hilbert order.

First, we generate DS candidate point cloud `is_shp` by ras chunks (which is a directory): 

In [ ]:
ds_can_is_shp_dir = './ds_processing/gix/ds_can_is_shp'
ds_can_key = './ds_processing/gix/ds_can_key.zarr'

In [ ]:
mc.ras2pc_ras_chunk(ds_can_gix,is_shp,ds_can_is_shp_dir,ds_can_key,chunks=(1000,1000))

Then we estimate the coherence matrix:

In [ ]:
ds_can_coh_dir = './ds_processing/gix/ds_can_coh'

In [ ]:
mc.emperical_co_pc(rslc,ds_can_is_shp_dir,ds_can_gix,ds_can_coh_dir,cuda=True)

The output of `emperical_co_pc` is point cloud data in the raster chunk order.
So we prepare the key to sort the point cloud data to hilbert order (the hillbert index of DS candidate can be generated by the way):

In [ ]:
ds_can_hix_unsorted = './ds_processing/gix/ds_can_hix.zarr'
ds_can_hix = './ds_processing/hix/ds_can/ds_can_hix.zarr'
gix2hix_key = './ds_processing/gix/gix2hix.zarr'

In [ ]:
mc.pc_hix(ds_can_gix, ds_can_hix_unsorted, shape=zarr.open(rslc,mode='r').shape[:2])
mc.pc_sort(
    ds_can_hix_unsorted, ds_can_hix,
    shape=zarr.open(rslc,mode='r').shape[:2],
    pc_in = './ds_processing/gix/ds_can_gix.zarr',
    pc = './ds_processing/hix/ds_can/ds_can_gix.zarr',
    key=gix2hix_key,
)

Then concat and sort `ds_can_coh` to hilbert order:

In [ ]:
ds_can_coh = './ds_processing/hix/ds_can/ds_can_coh.zarr'
ds_can_coh_ave = './ds_processing/hix/ds_can/ds_can_coh_ave.zarr'

In [ ]:
chunks = zarr.open(ds_can_gix,mode='r').chunks[0]
mc.pc_concat(ds_can_coh_dir,ds_can_coh,key=[ds_can_key,gix2hix_key],chunks=chunks)

In [ ]:
n_points = zarr.open(ds_can_coh,mode='r').shape[0]
mc.data_reduce(ds_can_coh,ds_can_coh_ave,map_func=np.abs,reduce_func=np.sum,post_map_func=lambda x: x/n_points)

In [ ]:
ds_can_coh_ave_zarr = zarr.open(ds_can_coh_ave,mode='r')
# average coherence matrix of the DS candidates: x is the secondary image, y the reference image
mc.view(np.abs(mr.uncompress_coh(ds_can_coh_ave_zarr[:])), label='average coherence', clim=(0, 1))

## Visualize adpative multi-looked interferograms

Visualize interferograms after adaptive multi-looking and compare with the raw 1 by 1 look interferograms:

In [ ]:
mc.pc_pyramid(
    './ds_processing/hix/ds_can/ds_can_coh.zarr',
    './ds_processing/hix/ds_can/ds_can_coh_pyramid',
    yx = './ds_processing/hix/ds_can/ds_can_gix.zarr',
    ras_resolution=2,
)

In [ ]:
rslc_zarr = zarr.open(rslc,mode='r')
tnet = mr.TempNet.from_bandwidth(rslc_zarr.shape[2])

In [ ]:
dates = './load_data/meta.toml'
raw_intf = mc.view('./load_data/rslc_pyramid', show='intf_all', dates=dates, label='Raw interferograms')
# the phase of the compressed coherence matrix, with sliders for the reference and secondary image
ds_can_intf = mc.view('./ds_processing/hix/ds_can/ds_can_coh_pyramid', show='coh',
                      label='Adaptively multilooked interferograms')
raw_intf + ds_can_intf

## Phase linking

To get rid of the closure phase, phase linking is applied here. EMI regularizes the coherence matrices whose magnitude is not positive definite (`regularize`, default true; decisions 0023 and 0026), which is the case for most candidates when the number of images approaches the number of independent looks of the SHPs:

In [ ]:
ds_can_coh = './ds_processing/hix/ds_can/ds_can_coh.zarr'
ds_can_ph = './ds_processing/hix/ds_can/ds_can_ph.zarr'

In [ ]:
mc.emi(ds_can_coh,ds_can_ph)

We can also estimate a temporal coherence for DS:

In [ ]:
ds_can_t_coh = './ds_processing/hix/ds_can/ds_can_t_coh.zarr'

In [ ]:
mc.ds_temp_coh(ds_can_coh,ds_can_ph, ds_can_t_coh)

Plot it:

In [ ]:
t_coh = './ds_processing/hix/ds_can/ras_ds_can_t_coh.zarr'
rslc_zarr = zarr.open(rslc,mode='r')
mc.pc2ras(ds_can_hix, ds_can_t_coh,t_coh,rslc_zarr.shape[:2])

In [ ]:
t_coh_zarr = zarr.open(t_coh,mode='r')
mc.view(t_coh, label='t_coh', clim=(0, 1))

The full coherence matrix may be too big to be saved.
It is also possible to do the coherence matrix estimation, phase linking together:

In [ ]:
ds_can_ph_dir = './ds_processing/gix/ds_can_ph'
ds_can_t_coh_dir = './ds_processing/gix/ds_can_t_coh'

In [ ]:
mc.emperical_co_emi_temp_coh_pc(rslc,ds_can_is_shp_dir,ds_can_gix,ds_can_ph_dir,ds_can_t_coh_dir,chunks=(1000,1000),cuda=True)
mc.pc_concat(
    [ds_can_ph_dir,ds_can_t_coh_dir],
    [ds_can_ph,ds_can_t_coh],
    key=[ds_can_key,gix2hix_key],
    chunks=200000)

## DS selection

Then we select the DS by their temporal coherence:

In [ ]:
t_coh_min = 0.8   # minimum temporal coherence of the phase linked DS
ds_hix = './ds_processing/hix/ds/ds_hix.zarr'
mc.pc_logic_pc(ds_can_hix, ds_can_t_coh, ds_hix, f'(pc_in>={t_coh_min})&(pc_in<=1.0)')

:::{.callout-note}
If too few points are selected, lower `t_coh_min`.
:::

## Visualize phase linked interferograms at refined ds points

In [ ]:
ds_coh = './ds_processing/hix/ds/ds_coh.zarr'
ds_ph = './ds_processing/hix/ds/ds_ph.zarr'
ds_gix = './ds_processing/hix/ds/ds_gix.zarr'

In [ ]:
mc.pc_select_data(
    ds_can_hix,ds_hix,
    pc_in=[ds_can_coh, ds_can_ph, './ds_processing/hix/ds_can/ds_can_gix.zarr/' ],
    pc=[ds_coh,ds_ph, ds_gix],threads_per_worker=32)

Then plot them:

In [ ]:
mc.pc_pyramid(
    './ds_processing/hix/ds/ds_coh.zarr',
    './ds_processing/hix/ds/ds_coh_pyramid',
    yx = './ds_processing/hix/ds/ds_gix.zarr',
    ras_resolution=2,
)

In [ ]:
mc.pc_pyramid(
    './ds_processing/hix/ds/ds_ph.zarr',
    './ds_processing/hix/ds/ds_ph_pyramid',
    yx = './ds_processing/hix/ds/ds_gix.zarr',
    ras_resolution=2,
)

In [ ]:
raw_intf = mc.view('./load_data/rslc_pyramid', show='intf_all', dates=dates, label='Raw interferograms')
ds_intf = mc.view('./ds_processing/hix/ds/ds_coh_pyramid', show='coh', label='Adaptively multilooked interferograms')
ds_pl_intf = mc.view('./ds_processing/hix/ds/ds_ph_pyramid', show='intf_all', label='Phase linked interferograms')
raw_intf + ds_intf + ds_pl_intf

## Visualize phase linked interferograms at refined ds points in earth coordinates

Get the needed data for ds:

In [ ]:
ds_ph = './ds_processing/hix/ds/ds_ph.zarr'
ds_e = './ds_processing/hix/ds/ds_e.zarr'
ds_n = './ds_processing/hix/ds/ds_n.zarr'

In [ ]:
mc.ras2pc(ds_hix,
          ras=['./load_data/e.zarr/','./load_data/n.zarr/'],
          pc=[ds_e, ds_n],)

In [ ]:
mc.pc_pyramid(
    './ds_processing/hix/ds/ds_ph.zarr',
    './ds_processing/hix/ds/ds_ph_geo_pyramid',
    x = ds_e,
    y = ds_n,
    ras_resolution=20,
)

In [ ]:
# web mercator coordinates: shown north up over a satellite base map
mc.view('./ds_processing/hix/ds/ds_ph_geo_pyramid', show='intf_all', dates='./load_data/meta.toml',
        label='Phase linked interferograms')